## Anime Recommendation System

In [ ]:
import pandas as pd
import numpy as np
from typing import List, Tuple, Optional
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.optim import Adam
import awswrangler as wr


%matplotlib inline

### reading AnimeList Dataset

In [ ]:
animelist_df = wr.s3.read_csv(
    path="s3://senpai-suggest-datasets/anime/animelist.csv",
    use_threads=True,
    usecols=["user_id", "anime_id", "rating"],
    nrows=1500,
)

In [ ]:
animelist_df.sample(5)

In [ ]:
# save the sample data reead from aws s3 to a local parquet file for future use
animelist_df.to_parquet("data/animelist_sample.parquet", index=False)

## User Ratings Dataset

### Data Ingestion

In [2]:
# read data from the S3 bucket but only read the first 1000 rows to avoid memory issues
def _ingest_data() -> pd.DataFrame:
    """
    Ingest data from an S3 CSV and return a pandas DataFrame.
    Returns:
        pd.DataFrame: A DataFrame containing the ingested data.
    Raises:
        RuntimeError: If there is an error reading the data from S3.
    """
    try:
        return wr.s3.read_csv(
            path="s3://your-bucket-name/your-file.csv",
            use_threads=True,  # must be keyword argument
            usecols=["user_id", "anime_id", "rating"],
            nrows=1000,
        )
    except Exception as exc:
        raise RuntimeError(f"Failed to read data from S3: {exc}") from exc


### Data Preprocessing

In [ ]:
def get_num_ratings(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Get the number of ratings for each user in the dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe containing user ratings.

    Returns:
    Tuple[pd.DataFrame, pd.Series]: A tuple containing the filtered dataframe and a series with user_id as index and number of ratings as values.
    """
    if "user_id" not in df.columns:
        raise ValueError("DataFrame must contain 'user_id' column.")
    n_ratings = df["user_id"].value_counts()
    df = df[df["user_id"].isin(n_ratings[n_ratings >= 400].index)].copy()
    return df, n_ratings


In [ ]:
def get_anime_stats(df: pd.DataFrame) -> Tuple[float, float, float]:
    """
    Get the minimum, maximum, and average rating from the dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe containing user ratings.

    Returns:
    Tuple[float, float, float]: A tuple containing minimum rating, maximum rating, and average rating.
    """
    if "rating" not in df.columns:
        raise ValueError("DataFrame must contain 'rating' column.")

    min_rating = min(df["rating"])
    max_rating = max(df["rating"])
    avg_rating = np.mean(df["rating"])

    return min_rating, max_rating, avg_rating

In [ ]:
def normalize_ratings(
    rating_df: pd.DataFrame,
    min_rating: float,
    max_rating: float,
) -> pd.DataFrame:
    """
    Normalize the ratings in the dataframe to a range of 0 to 1.

    Parameters:
    rating_df (pd.DataFrame): The input dataframe containing user ratings.
    min_rating (float): The minimum rating value.
    max_rating (float): The maximum rating value.

    Returns:
    pd.DataFrame: The dataframe with normalized ratings.
    """
    if "rating" not in rating_df.columns:
        raise ValueError("DataFrame must contain 'rating' column.")

    scale = max_rating - min_rating
    if scale == 0:
        raise ValueError("max_rating and min_rating cannot be the same.")

    rating_df["rating"] = ((rating_df["rating"] - min_rating) / scale).astype(np.float64)
    return rating_df

In [ ]:
def check_duplicates(df: pd.DataFrame) -> bool:
    """
    Check for duplicate rows in the dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe to check for duplicates.

    Returns:
    bool: True if duplicates are found, False otherwise.
    """
    return df.duplicated().any()


def check_nulls(df: pd.DataFrame) -> bool:
    """
    Check for null values in the dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe to check for null values.

    Returns:
    bool: True if null values are found, False otherwise.
    """
    return df.isnull().values.any()

In [ ]:
def encode_users(df: pd.DataFrame) -> Tuple[pd.DataFrame, dict, dict]:
    """
    Encode user IDs in the dataframe to a continuous range of integers.

    Parameters:
    df (pd.DataFrame): The input dataframe containing user ratings.

    Returns:
    Tuple[pd.DataFrame, dict, dict]: A tuple containing the dataframe with encoded user IDs,
                                    a dictionary mapping original user IDs to encoded IDs,
                                    and a dictionary mapping encoded IDs back to original user IDs.
    """
    if "user_id" not in df.columns:
        raise ValueError("DataFrame must contain 'user_id' column.")

    user_ids = df["user_id"].unique().tolist()
    user2user_encoded = {user_id: i for i, user_id in enumerate(user_ids)}
    user2user_decoded = {i: user_id for i, user_id in enumerate(user_ids)}
    # create a new column in the dataframe with the encoded user IDs(from userid = 123456 to user = 0)
    df["user"] = df["user_id"].map(user2user_encoded)

    return df, user2user_encoded, user2user_decoded


In [ ]:
def reset_user_df_indexes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reset the index of the dataframe and drop the old index.

    Parameters:
    df (pd.DataFrame): The input dataframe to shuffle and reset the index.

    Returns:
    pd.DataFrame: The dataframe with reset index.
    """
    return df.sample(frac=1).reset_index(drop=True)

### Ratings Preprocessing Pipeline

In [ ]:
""" create a preprocessing pipeline class for the anime data
1. get_num_ratings
2. get_anime_stats - exclude this step
3. normalize_ratings
4. check_duplicates
5. check_nulls
6. encode_users
7. reset_user_df_indexes
8. Save the preprocessed data to a parquet file in the data folder  and s3 bucket
"""


class Preprocessing:
    """class for creating preprocessing steps for the anime data"""

    @staticmethod
    def get_num_ratings(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        """
            Get the number of ratings for each user in the dataframe.

        Parameters:
            df (pd.DataFrame): The input dataframe containing user ratings.

        Returns:
        Tuple[pd.DataFrame, pd.Series]: A tuple containing the filtered dataframe and a series with user_id as index and number of ratings as values.
        """
        if "user_id" not in df.columns:
            raise ValueError("DataFrame must contain 'user_id' column.")
        n_ratings = df["user_id"].value_counts()
        df = df[df["user_id"].isin(n_ratings[n_ratings >= 400].index)].copy()
        return df, n_ratings

    @staticmethod
    def normalize_ratings(
        rating_df: pd.DataFrame,
        min_rating: float,
        max_rating: float,
    ) -> pd.DataFrame:
        """
        Normalize the ratings in the dataframe to a range of 0 to 1.

        Parameters:
            rating_df (pd.DataFrame): The input dataframe containing user ratings.
            min_rating (float): The minimum rating value.
            max_rating (float): The maximum rating value.

        Returns:
            pd.DataFrame: The dataframe with normalized ratings.
        """
        if "rating" not in rating_df.columns:
            raise ValueError("DataFrame must contain 'rating' column.")

        scale = max_rating - min_rating
        if scale == 0:
            raise ValueError("max_rating and min_rating cannot be the same.")

        rating_df["rating"] = ((rating_df["rating"] - min_rating) / scale).astype(np.float64)
        return rating_df

    @staticmethod
    def check_duplicates(df: pd.DataFrame) -> bool:
        """
        Check for duplicate rows in the dataframe.

        Parameters:
            df (pd.DataFrame): The input dataframe to check for duplicates.

        Returns:
            bool: True if duplicates are found, False otherwise.
        """
        return df.duplicated().any()

    @staticmethod
    def check_nulls(df: pd.DataFrame) -> bool:
        """
        Check for null values in the dataframe.

        Parameters:
            df (pd.DataFrame): The input dataframe to check for null values.

        Returns:
            bool: True if null values are found, False otherwise.
        """
        return df.isnull().any().any()

    @staticmethod
    def encode_users(df: pd.DataFrame) -> Tuple[pd.DataFrame, dict, dict]:
        """
        Encode user IDs in the dataframe to a continuous range of integers.

        Parameters:
            df (pd.DataFrame): The input dataframe containing user ratings.
        Returns:
            Tuple[pd.DataFrame, dict, dict]: A tuple containing the dataframe with encoded user IDs, a dictionary mapping original user IDs to encoded IDs, and a dictionary mapping encoded IDs back to original user IDs.
        """
        if "user_id" not in df.columns:
            raise ValueError("DataFrame must contain 'user_id' column.")

        user_mapping = {user_id: idx for idx, user_id in enumerate(df["user_id"].unique())}
        df["user_id"] = df["user_id"].map(user_mapping)
        reverse_mapping = {idx: user_id for user_id, idx in user_mapping.items()}
        return df, user_mapping, reverse_mapping

    @staticmethod
    def reset_user_df_indexes(df: pd.DataFrame) -> pd.DataFrame:
        """
        Reset the index of the dataframe and drop the old index.

        Parameters:
            df (pd.DataFrame): The input dataframe to shuffle and reset the index.

        Returns:
            pd.DataFrame: The dataframe with reset indexes.
        """
        return df.sample(frac=1).reset_index(drop=True)

    @staticmethod
    def save_to_parquet(df: pd.DataFrame, local_path: str, s3_path: str) -> None:
        """
        Save the dataframe to a local parquet file and upload it to an S3 bucket.

        Parameters:
            df (pd.DataFrame): The input dataframe to save.
            local_path (str): The local file path to save the parquet file.
            s3_path (str): The S3 bucket path to upload the parquet file.
        """
        df.to_parquet(local_path, index=False)
        wr.s3.to_parquet(df=df, path=s3_path, index=False)


## Anime dataset

In [ ]:
# ingest animee csv data from s3 bucket and save it to a local parquet file for future use
def _ingest_anime_data() -> pd.DataFrame:
    """
    Ingest data from an S3 CSV and return a pandas DataFrame.
    Returns:
        pd.DataFrame: A DataFrame containing the ingested data.
    Raises:
        RuntimeError: If there is an error reading the data from S3.
    """
    try:
        return wr.s3.read_csv(
            path="s3://your-bucket-name/your-file.csv",
            nrows=1000,
        )
    except Exception as exc:
        raise RuntimeError(f"Failed to read data from S3: {exc}") from exc

In [ ]:
def removing_unknowns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace rows with "unknown" with NaN in the dataframe.

    Parameters:
        df (pd.DataFrame): The input dataframe to clean.
    Returns:
        pd.DataFrame: The cleaned dataframe with "unknown" values removed.
    """
    if df.empty:
        raise ValueError("DataFrame is empty. Cannot remove unknowns from an empty DataFrame.")

    # Replace 'unknown' with NaN (case insensitive)
    df.replace(to_replace=r"(?i)unknown", value=pd.NA, inplace=True)

    return df

In [ ]:
# get the original names of the animeee by id and geet the english version of the name if available, else return the original name
def get_anime_name_by_id(anime_df: pd.DataFrame, anime_id: int) -> str:
    """
    Get the original name of the anime by its ID. If an English version of the name is available, return that; otherwise, return the original name.

    Parameters:
        anime_df (pd.DataFrame): The dataframe containing anime information.
        anime_id (int): The ID of the anime to look up.

    Returns:
        str: The name of the anime.
    """
    if "anime_id" not in anime_df.columns:
        raise ValueError("DataFrame must contain 'anime_id' column.")

    try:
        name = anime_df[anime_df.anime_id == anime_id].eng_version.values[0]
        if name is np.nan:
            name = anime_df[anime_df.anime_id == anime_id].name.values[0]
        return name
    except IndexError:
        raise ValueError(f"No anime found with ID {anime_id}.")


def add_english_name_column(
    anime_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add a new column 'eng_name' to the anime dataframe that contains the English version of the anime name if available, otherwise the original name.

    Parameters:
        anime_df (pd.DataFrame): The dataframe containing anime information.
    Returns:
        pd.DataFrame: The updated dataframe with the new 'eng_name' column.
    """
    # create the column
    anime_df["anime_id"] = anime_df["MAL_ID"]
    anime_df["eng_version"] = anime_df["English_name"]
    anime_df["eng_version"] = anime_df.apply(
        lambda row: get_anime_name_by_id(anime_df, row["anime_id"]), axis=1
    )
    return anime_df

In [ ]:
# sort the anime list by score
def sort_anime_by_score(anime_df: pd.DataFrame) -> pd.DataFrame:
    """
    Sort the anime dataframe by score in descending order.

    Parameters:
        anime_df (pd.DataFrame): The dataframe containing anime information.
    Returns:
        pd.DataFrame: The sorted dataframe by score.
    """
    if "Score" not in anime_df.columns:
        raise ValueError("DataFrame must contain 'Score' column.")

    return anime_df.sort_values(by="Score", ascending=False, kind="quicksort").reset_index(
        drop=True
    )

In [ ]:
# filter the anime list to only include the columns we need
def filter_anime_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter the anime dataframe to only include the necessary columns.

    Parameters:
        df (pd.DataFrame): The dataframe containing anime information.
    Returns:
        pd.DataFrame: The filtered dataframe with only the necessary columns.
    """
    required_columns = [
        "anime_id",
        "eng_version",
        "Score",
        "Genres",
        "Episodes",
        "Type",
        "Premiered",
        "Members",
    ]

    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"DataFrame is missing required columns: {missing_columns}")
    return df[
        ["anime_id", "eng_version", "Score", "Genres", "Episodes", "Type", "Premiered", "Members"]
    ]

In [ ]:
# get the anime dataframe for a specific anime by ID or English name e.g. get_anime_frame("Steins;Gate", df) or get_anime_frame(40028, df)
def get_anime_frame(anime: int | str, df: pd.DataFrame) -> pd.DataFrame:
    """Get the anime dataframe for a specific anime by ID or English name.

    Args:
        anime (int | str): The anime ID (int) or English name (str).
        df (pd.DataFrame): The dataframe containing anime information.

    Returns:
        pd.DataFrame: The dataframe containing the specific anime.
    """
    if isinstance(anime, int):
        return df[df.anime_id == anime]
    if isinstance(anime, str):
        return df[df.eng_version == anime]
    raise ValueError(f"Anime must be an int (ID) or str (English name), got {type(anime)}")

### Data Quality Checks with Great Expectations

In [ ]:
# get a sample of the ratings data
from typing import Optional


def get_sample_ratings(
    df: pd.DataFrame,
    sample_size: Optional[int] = 1000,
) -> pd.DataFrame:
    """
    Get a random sample of the ratings data.

    Parameters:
        df (pd.DataFrame): The input dataframe containing user ratings.
        sample_size (int): The number of samples to return. Default is 1000.

    Returns:
        pd.DataFrame: A dataframe containing a random sample of the ratings data.
    """
    if not isinstance(sample_size, int) or sample_size <= 0:
        raise ValueError("sample_size must be a positive integer.")

    if sample_size is None:
        return df.sample(frac=1, random_state=42).reset_index(drop=True)
    if sample_size > len(df):
        raise ValueError("sample_size cannot be greater than the number of rows in the dataframe.")
    return df.sample(n=sample_size, random_state=42).reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split


# get the dependent and independent variables for the model
def get_features_and_target(df: pd.DataFrame) -> Tuple[np.ndarray, pd.Series]:
    """
    Get the independent variables (features) and dependent variable (target) from the dataframe.

    Parameters:
        df (pd.DataFrame): The input dataframe containing user ratings.
    Returns:
        Tuple[np.ndarray, pd.Series]: A tuple containing the features array and the target series.
    """
    if "rating" not in df.columns:
        raise ValueError("DataFrame must contain 'rating' column.")

    X = df[["user", "anime_id"]].values
    y = df["rating"]
    return X, y


def split_train_test(
    X: np.ndarray,
    y: pd.Series,
    test_size: float = 0.2,
    random_state: int = 42,
) -> Tuple[np.ndarray, np.ndarray, pd.Series, pd.Series]:
    """
    Split the features and target into training and testing sets.

    Parameters:
        X (np.ndarray): The features array.
        y (pd.Series): The target series.
        test_size (float): The proportion of the dataset to include in the test split. Default is 0.2.
        random_state (int): The seed used by the random number generator. Default is 42.

    Returns:
        Tuple[np.ndarray, np.ndarray, pd.Series, pd.Series]: A tuple containing the training features, testing features, training target, and testing target.
    """
    if not isinstance(test_size, float) or not (0 < test_size < 1):
        raise ValueError("test_size must be a float between 0 and 1.")

    if not isinstance(random_state, int):
        raise ValueError("random_state must be an integer.")

    return train_test_split(X, y, test_size=test_size, random_state=random_state)